# Libraries

In [1]:
import os
import sys

root_models = '/kaggle/input/models/rickmark369/symbolic-regression-thesis-magistrale-unipi/other/default/18'

sys.path.insert(0, root_models)
sys.path.insert(0, "/kaggle/input/datasets")

root_dataset = '/kaggle/input/datasets/rickmark369/tesi-magistrale-unipi/'
root_models  = root_models + '/'


root_dataset = ""
root_models = ""

In [2]:
#%pip install lark graphviz xicorpy pmlb

In [3]:
import pmlb
import pandas as pd
import numpy as np
import time
import yaml
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import json

import LearningStrategy as ls
from Model import SymbolicRegression
import Errors as er
import Indicators as ind

# Prepare datasets

In [4]:
cavalab_bb_results = pd.read_feather(root_dataset + 'InformationSOTA/srbench-master/results/black-box_results.feather')
cavalab_gt_results = pd.read_feather(root_dataset + 'InformationSOTA/srbench-master/results/ground-truth_results.feather')

print(cavalab_bb_results.shape)
print(cavalab_gt_results.shape)

bb_dataset_names = [i for i in cavalab_bb_results["dataset"].unique()]
gt_dataset_names = [i for i in cavalab_gt_results["dataset"].unique()]

cavalab_bb_results.head()

(25346, 14)
(69088, 28)


,dataset,algorithm,random_state,training time (s),model_size,symbolic_model,mse_test,mae_test,r2_test,params_str,training time (hr),r2_zero_test,friedman_dataset,symbolic_alg
0,1027_ESL,AdaBoost,11284,32.140310,14962,not implemented,0.474719,0.524467,0.765888,"{'learning_rate': 0.01, 'loss': 'linear', 'n_e...",0.008928,0.765888,False,False
1,1027_ESL,AdaBoost,11964,30.652193,14986,not implemented,0.364587,0.460950,0.828411,"{'learning_rate': 0.01, 'loss': 'linear', 'n_e...",0.008514,0.828411,False,False
2,1027_ESL,AdaBoost,15795,25.045438,3176,not implemented,0.343520,0.457305,0.833438,"{'learning_rate': 0.1, 'loss': 'linear', 'n_es...",0.006957,0.833438,False,False
3,1027_ESL,AdaBoost,21575,31.495630,3223,not implemented,0.486905,0.523346,0.813623,"{'learning_rate': 0.1, 'loss': 'linear', 'n_es...",0.008749,0.813623,False,False
4,1027_ESL,AdaBoost,22118,21.049579,14992,not implemented,0.317529,0.419290,0.830149,"{'learning_rate': 0.01, 'loss': 'linear', 'n_e...",0.005847,0.830149,False,False


In [5]:
print(cavalab_bb_results.columns)
print("-"*100)
print(cavalab_gt_results.columns)

print('\n'*3)

models_bb = sorted([i for i in cavalab_bb_results["algorithm"].unique()])
models_gt = sorted([i for i in cavalab_gt_results["algorithm"].unique()])


print(models_bb)
print(models_gt)

print(sorted(list(set(models_bb).intersection(models_gt))))

print('\n'*3)

print(bb_dataset_names)
print(gt_dataset_names)

Index(['dataset', 'algorithm', 'random_state', 'training time (s)',
       'model_size', 'symbolic_model', 'mse_test', 'mae_test', 'r2_test',
       'params_str', 'training time (hr)', 'r2_zero_test', 'friedman_dataset',
       'symbolic_alg'],
      dtype='object')
----------------------------------------------------------------------------------------------------
Index(['dataset', 'algorithm', 'random_state', 'process_time',
       'training time (s)', 'target_noise', 'feature_noise', 'true_model',
       'model_size', 'symbolic_model', 'mse_train', 'mae_train', 'r2_train',
       'mse_test', 'mae_test', 'r2_test', 'simplified_symbolic_model',
       'simplified_complexity', 'symbolic_error', 'symbolic_fraction',
       'symbolic_error_is_zero', 'symbolic_error_is_constant',
       'symbolic_fraction_is_constant', 'sympy_exception',
       'training time (hr)', 'r2_zero_test', 'data_group',
       'symbolic_solution'],
      dtype='object')




['AFP', 'AFP_FE', 'AIFeynman', 'AdaBoos

In [6]:
#Calculate the seeds used in every datasets
dict_seeds = {}

for name in bb_dataset_names:
    dict_seeds[name] = set([x for x in cavalab_bb_results.loc[cavalab_bb_results["dataset"] == name, "random_state"].unique()])
    
for name in gt_dataset_names:
    dict_seeds[name] = set([x for x in cavalab_gt_results.loc[cavalab_gt_results["dataset"] == name, "random_state"].unique()])

total_set = set([x for x in cavalab_bb_results["random_state"].unique()]).union(set([x for x in cavalab_gt_results["random_state"].unique()]))

for key in dict_seeds.keys():
    
    total_set = total_set.intersection(dict_seeds[key])
    
print(total_set)

{15795, 860, 23654, 5390}


In [7]:
def split(x, perc1, perc2):
    train_len = int(perc1*len(x))
    val_len = int((perc1+perc2)*len(x))
    
    #TR, VL, TS
    return x[:train_len], x[train_len:val_len], x[val_len:]

def is_prefix(name, prefix):
    return name[:len(prefix)] == prefix

def adapt_name(name):
    
    expection_names = {
        "feynman_I_48_2": "feynman_I_48_20",
                       }
    
    if(name in expection_names.keys()): name = expection_names[name]
    
    if(is_prefix(name, "feynman_test")): return name[len('feynamn_'):]
    elif(is_prefix(name, "feynman")): return name[len('feynamn_'):].replace('_', '.')
    elif(is_prefix(name, "strogatz")): return name.replace('strogatz', 'd')



shuffle_dataset_seed = 860  #Form the 5 universal seed, we choose 860


#Si ridà direttamente il dataset splittato: si fa uno split hold out del 75 di training set e 25% di test set (lo dice alla fine della sezione 3.2 del paper Call for action, tipo preprint).
#Da capire il seed quali sono.
def get_dataset(name):

    if(name in bb_dataset_names):
    
        x, y = pmlb.fetch_data(name, return_X_y=True)
        
        x = x.T

        result = {"x_dict": {f"X{i}": x[i] for i in range(x.shape[0])}, "y": y, "formula": None}
    
    elif(name in gt_dataset_names):
        
        adapted_name = adapt_name(name)
        
        formulas = pd.read_csv(root_dataset + "InformationSOTA/Equations.csv")
        
        try: formula_founded = formulas.loc[formulas["Filename"] == adapted_name, "Formula"].tolist()[0]
        except IndexError: formula_founded = None
        
        if(is_prefix(name, "feynman_test")): file = root_dataset + f"InformationSOTA/Feynman_bonus_with_units/{adapted_name}"
        elif(is_prefix(name, "feynman")): file =    root_dataset + f"InformationSOTA/Feynman_with_units/{adapted_name}"
        elif(is_prefix(name, "strogatz")): file =   root_dataset + f"InformationSOTA/ode-strogatz-master/{adapted_name}.txt"
        
        if(is_prefix(name, "feynman")):
            
            feature_names = [x["name"] for x in yaml.safe_load(open(root_dataset + f"InformationSOTA/pmlb-master/datasets/{name}/metadata.yaml", "r"))["features"]]
            
            table = np.loadtxt(file).T
            result = {"x_dict": {name: table[i] for name, i in zip(feature_names, range(table.shape[0]-1))}, "y": table[-1], "formula": formula_founded}
        
        else:
            
            table = pd.read_csv(file)
            result = {"x_dict": {l: table[l].to_numpy() for l in table.columns.tolist() if l != "label"}, "y": table["label"].to_numpy(), "formula": formula_founded}
            
        
    else: raise ValueError(f"Dataset {name} are not aviable")
     
    if(np.any(~np.isfinite(result["y"])) or any(np.any(~np.isfinite(result["x_dict"][k])) for k in result["x_dict"].keys())):
        raise ValueError("Some data are not finite value, the dataset is not valid")
    
    
    for k in result["x_dict"].keys(): result["x_dict"][k] = result["x_dict"][k].astype(np.float64)
    result["y"] = result["y"].astype(np.float64)
        
    #Prepare the hyper-parameters choosed for this experiments

    #Prepare the slots for split
    result["x_dict train"] = {}
    result["x_dict val"] = {}
    result["x_dict test"] = {}
    
    #Prepare the shuffler
    shuffler = np.random.default_rng(seed = shuffle_dataset_seed)
    
    indexes = np.arange(len(result["y"]))
    shuffler.shuffle(indexes)

    TR_perc = 0.60
    VL_perc = 0.15
    
    #Split all X
    for key in result["x_dict"].keys():
        result["x_dict train"][key], result["x_dict val"][key], result["x_dict test"][key] = split(result["x_dict"][key][indexes], TR_perc, VL_perc)
    
    #Split Y
    result["y train"], result["y val"], result["y test"] = split(result["y"][indexes], TR_perc, VL_perc)
    
    #Delete old Y and all X
    del result["x_dict"]
    del result["y"]

    return result

In [8]:
def ParallelExecution(threads, f_to_execute, list_to_apply_execution):
    return list(ThreadPoolExecutor(max_workers=threads).map(f_to_execute, list_to_apply_execution))

In [9]:
def calculate_dimension(dataset_name):

    print(dataset_name, "execution")

    try:

        #Extract inputs and output accordind to the dataset name
        dataset = get_dataset(dataset_name)
        TR_inputs_train = dataset["x_dict train"]
        TR_output_train = dataset["y train"]
        VL_output_train = dataset["y val"]
        TS_output_test = dataset["y test"]
    
        return {'dataset': dataset_name,
                'features': len(TR_inputs_train.keys()),
                'TR points': len(TR_output_train),
                'VL points': len(VL_output_train),
                'TS points': len(TS_output_test)}

    except: return {'dataset': dataset_name}

dataset_names = bb_dataset_names + gt_dataset_names

results = []

#results = ParallelExecution(4, calculate_dimension, dataset_names)
#json.dump(results, open(f"DatasetDimension.json", "w", encoding="utf-8"), indent=4, ensure_ascii=False)

In [10]:
datasets_dimension = json.load(open("DatasetDimension.json", "r"))

sorting_datasets = [(x["dataset"], x["features"]*x["TR points"]) for x in datasets_dimension if "features" in x.keys()]
sorting_datasets = sorted(sorting_datasets, key = lambda r: r[1])

names = ["'" + i[0] + "'" for i in sorting_datasets]

#print("[" + ",\n".join(names) + "]")

# Prepare tools for training

In [11]:
class Counter:
        
    def __init__(self, i):
        self.__i = i
        self.__r = 0
            
    def __call__(self):
        self.__r += 1
        return self.__r > self.__i


def execute(dataset_name):
    
    if(os.path.exists(f"BenchmarkResults/{dataset_name}.json")):
        print(f"{dataset_name} SKIP")
        return
    
    print(f"Execute {dataset_name}")

    #try:

    #Extract inputs and output accordind to the dataset name
    dataset = get_dataset(dataset_name)
    TR_inputs_train = dataset["x_dict train"]
    TR_output_train = dataset["y train"]

    counter = Counter(250)
            
    tree = SymbolicRegression.string_to_tree("0")
    
    learning_algorithm = ls.GreedyStatisticalMapping()
            
    metric_results = tree.learn(TR_inputs_train,
                                TR_output_train,
                                learning_algorithm,
                                stop_criteria = lambda ls: counter(),
                                metric_functions = {"tree": lambda model: str(model), "timestamp": lambda model: time.time()})
            
    if(not os.path.exists("BenchmarkResults")): os.mkdir("BenchmarkResults")
               
    json.dump(metric_results, open(f"BenchmarkResults/{dataset_name}.json", "w", encoding="utf-8"), indent=4, ensure_ascii=False)
    print(f"Results of dataset {dataset_name} saved!")
    
    #except: print(f"Error on training model with {dataset_name} dataset")

# Experiments

In [12]:
dataset_to_do = [
'192_vineyard',
'228_elusage',
'485_analcatdata_vehicle',
'1096_FacultySalaries',
'523_analcatdata_neavote',
'663_rabe_266',
'687_sleuth_ex1605',
'659_sleuth_ex1714',
'678_visualizing_environmental',
'712_chscase_geyser1',
'594_fri_c2_100_5',
'611_fri_c3_100_5',
'624_fri_c0_100_5',
'656_fri_c1_100_5',
'210_cloud',
'706_sleuth_case1202',
'1089_USCrime',
'519_vinnie',
'strogatz_bacres1',
'strogatz_bacres2',
'strogatz_barmag1',
'strogatz_barmag2',
'strogatz_glider1',
'strogatz_glider2',
'strogatz_lv1',
'strogatz_lv2',
'strogatz_predprey1',
'strogatz_predprey2',
'strogatz_shearflow1',
'strogatz_shearflow2',
'strogatz_vdp1',
'strogatz_vdp2',
'665_sleuth_case2002',
'542_pollution',
'527_analcatdata_election2000',
'591_fri_c1_100_10',
'621_fri_c0_100_10',
'634_fri_c2_100_10',
'230_machine_cpu',
'579_fri_c0_250_5',
'596_fri_c2_250_5',
'601_fri_c1_250_5',
'613_fri_c3_250_5',
'690_visualizing_galaxy',
'556_analcatdata_apnea2',
'557_analcatdata_apnea1',
'561_cpu',
'1027_ESL',
'229_pwLinear',
'597_fri_c2_500_5',
'602_fri_c3_250_10',
'615_fri_c4_250_10',
'617_fri_c3_500_5',
'631_fri_c1_500_5',
'635_fri_c0_250_10',
'647_fri_c1_250_10',
'649_fri_c0_500_5',
'651_fri_c0_100_25',
'657_fri_c2_250_10',
'695_chatfield_4',
'522_pm10',
'547_no2',
'560_bodyfat',
'1029_LEV',
'1030_ERA',
'599_fri_c2_1000_5',
'604_fri_c4_500_10',
'609_fri_c0_1000_5',
'612_fri_c1_1000_5',
'627_fri_c2_500_10',
'628_fri_c3_1000_5',
'641_fri_c1_500_10',
'646_fri_c3_500_10',
'654_fri_c0_500_10',
'666_rmftsa_ladata',
'605_fri_c2_250_25',
'644_fri_c4_250_25',
'653_fri_c0_250_25',
'658_fri_c3_250_25',
'1028_SWD',
'593_fri_c1_1000_10',
'595_fri_c0_1000_10',
'606_fri_c2_1000_10',
'608_fri_c3_1000_10',
'623_fri_c4_1000_10',
'banana',
'581_fri_c3_500_25',
'582_fri_c1_500_25',
'584_fri_c4_500_25',
'603_fri_c0_250_50',
'633_fri_c0_500_25',
'643_fri_c2_500_25',
'648_fri_c1_250_50',
'529_pollen',
'titanic',
'586_fri_c3_1000_25',
'589_fri_c2_1000_25',
'592_fri_c4_1000_25',
'598_fri_c0_1000_25',
'616_fri_c4_500_50',
'620_fri_c1_1000_25',
'626_fri_c2_500_50',
'637_fri_c1_500_50',
'645_fri_c3_500_50',
'650_fri_c0_500_50',
'505_tecator',
'583_fri_c1_1000_50',
'590_fri_c0_1000_50',
'607_fri_c4_1000_50',
'618_fri_c3_1000_50',
'622_fri_c2_1000_50',
'225_puma8NH',
'503_wind',
'227_cpu_small',
'562_cpu_small',
'588_fri_c4_1000_100',
'4544_GeographicalOriginalofMusic',
'1199_BNG_echoMonths',
'537_houses',
'197_cpu_act',
'573_cpu_act',
'218_house_8L',
'294_satellite_image',
'1193_BNG_lowbwt',
'574_house_16H',
'215_2dplanes',
'344_mv',
'564_fried',
'201_pol',
'feynman_I_6_2a',
'1201_BNG_breastTumor',
'1203_BNG_pwLinear',
'feynman_III_12_43',
'feynman_II_11_28',
'feynman_II_27_18',
'feynman_II_38_14',
'feynman_II_3_24',
'feynman_II_8_31',
'feynman_I_12_1',
'feynman_I_12_5',
'feynman_I_14_4',
'feynman_I_25_13',
'feynman_I_29_4',
'feynman_I_34_27',
'feynman_I_39_1',
'feynman_I_6_2',
'feynman_III_15_12',
'feynman_III_15_14',
'feynman_III_15_27',
'feynman_III_17_37',
'feynman_III_7_38',
'feynman_III_8_54',
'feynman_II_10_9',
'feynman_II_13_23',
'feynman_II_13_34',
'feynman_II_15_4',
'feynman_II_15_5',
'feynman_II_24_17',
'feynman_II_27_16',
'feynman_II_34_2',
'feynman_II_34_29a',
'feynman_II_37_1',
'feynman_II_4_23',
'feynman_II_8_7',
'feynman_I_10_7',
'feynman_I_12_4',
'feynman_I_14_3',
'feynman_I_15_10',
'feynman_I_16_6',
'feynman_I_18_12',
'feynman_I_27_6',
'feynman_II_34_2a',
'feynman_I_30_3',
'feynman_I_34_1',
'feynman_I_34_14',
'feynman_I_37_4',
'feynman_I_39_11',
'feynman_I_43_31',
'feynman_I_47_23',
'feynman_I_48_2',
'feynman_I_6_2b',
'feynman_III_10_19',
'feynman_III_13_18',
'feynman_III_21_20',
'feynman_III_4_32',
'feynman_III_4_33',
'feynman_II_11_27',
'feynman_II_13_17',
'feynman_II_34_11',
'feynman_II_38_3',
'feynman_II_6_11',
'feynman_II_6_15b',
'feynman_I_13_4',
'feynman_I_15_3t',
'feynman_I_15_3x',
'feynman_I_18_14',
'feynman_I_18_4',
'feynman_I_24_6',
'feynman_I_29_16',
'feynman_I_12_2',
'feynman_I_32_5',
'feynman_I_34_8',
'feynman_I_38_12',
'feynman_I_39_22',
'feynman_I_43_16',
'feynman_I_43_43',
'feynman_I_50_26',
'feynman_I_8_14',
'feynman_test_11',
'feynman_test_15',
'feynman_test_3',
'feynman_test_5',
'feynman_test_8',
'feynman_III_14_14',
'feynman_III_19_51',
'feynman_II_11_3',
'feynman_II_21_32',
'feynman_II_2_42',
'feynman_II_34_29b',
'feynman_II_35_18',
'feynman_II_35_21',
'feynman_I_12_11',
'feynman_I_13_12',
'feynman_II_11_20',
'feynman_I_44_4',
'feynman_I_41_16',
'feynman_test_12',
'feynman_test_13',
'feynman_test_14',
'feynman_test_18',
'feynman_test_4',
'feynman_test_7',
'feynman_test_9',
'feynman_III_9_52',
'feynman_II_6_15a',
'feynman_I_11_19',
'feynman_test_19',
'feynman_I_32_17',
'feynman_I_40_1',
'feynman_test_16',
'feynman_test_17',
'feynman_test_2',
'feynman_test_1',
'feynman_test_20',
'feynman_test_6',
'feynman_II_36_38',
'feynman_I_9_18',
'1196_BNG_pharynx',
'1595_poker',
'1191_BNG_pbc']

dataset_done = []

In [13]:
#import os

#names = ["'" + i.split(".")[0] + "'" for i in os.listdir("BenchmarkResults")]
#print("[" + ",\n".join(names) + "]")

In [14]:
#for x in dataset_to_do:
    #if x not in dataset_done: execute(x)

# Calculate errors on datasets

In [15]:
def linear_extension(X, Y):
    
    mean_X = np.mean(X)
    mean_Y = np.mean(Y)
        
    m = np.sum((X - mean_X)*(Y - mean_Y)) / np.sum((X - mean_X)**2)
    q = mean_Y - (m*mean_X)
    
    return m*X + q


error_metrics = {"MSE": lambda y_model, y_true: er.MSE(y_model, y_true),
                 "MAE": lambda y_model, y_true: er.MAE(y_model, y_true),
                 "R2": lambda  y_model, y_true: er.R2(y_model, y_true)}



def apply(results, tree, TR_inputs, TR_output, label):

    try: output_model = tree.get_root().forward_calculation_memory_sub_tree(TR_inputs).get_memory("forward")
    except: output_model = np.nan
    
    output_model_lin_ext = linear_extension(output_model, TR_output)
    
    for metric in error_metrics.keys(): 
        
        internal_label = f"{label} {metric}"
        if(internal_label not in results): results[internal_label].append(error_metrics[metric](output_model, TR_output))
        
        internal_label = f"{label} {metric} lin. ext."
        if(internal_label not in results): results[internal_label].append(error_metrics[metric](output_model_lin_ext, TR_output))
    
    

def calculate_error(dataset_name):

    print(f"Calculate errors about {dataset_name} dataset")

    experiment = json.load(open(root_models + f"/BenchmarkResults/{dataset_name}.json", "r"))

    #Extract inputs and output accordind to the dataset name
    dataset = get_dataset(dataset_name)
    
    TR_inputs = dataset["x_dict train"]
    TR_output = dataset["y train"]
    
    VL_inputs = dataset["x_dict val"]
    VL_output = dataset["y val"]
    
    errors = {"Nodes" : []}

    for step in experiment["results"]["tree"]:
        
        tree = SymbolicRegression.string_to_tree(step)
        
        errors["Nodes"].append(len(tree.get_root().get_all_subtree_nodes()))

        apply(errors, tree, TR_inputs, TR_output, "TR")
        apply(errors, tree, VL_inputs, VL_output, "VL")
      
    if(not os.path.exists("BenchmarkCompleteResults")): os.mkdir("BenchmarkCompleteResults")

    experiment["results"] |= errors
            
    json.dump(experiment, open(f"BenchmarkCompleteResults/{dataset_name}.json", "w", encoding="utf-8"), indent=4, ensure_ascii=False)
    print(f"Results of adding errors {dataset_name} saved!")   

#ParallelExecution(4, calculate_error, dataset_already_done)

In [17]:
def add_information_performance(result, index, label, models, TS_MAE, TS_MSE, TS_R2, Nodes):
    
    result |= {f"{label} model":     models[index],
              f"complexity {label}": Nodes[index],
              f"MAE TS {label}":     TS_MAE[index],
              f"MSE TS {label}":     TS_MSE[index],
              f"R2 TS {label}":       TS_R2[index]}


def informations(dataset_name):
    
    model_result = json.load(open(f"BenchmarkCompleteResults/{dataset_name}.json", "r"))
    
    ground_truth = get_dataset(dataset_name)["formula"]
    
    last_model_index = -1    

    models =  model_result["results"]["tree"]
    VL_list = model_result["results"]["VL MAE"]
    
    while(not np.isfinite(VL_list[last_model_index])): last_model_index -= 1
    
    best_model_index = VL_list.index(min([x for x in VL_list if np.isfinite(x)]))
    
    TS_MAE = model_result["results"]["TS MAE"]
    TS_MSE = model_result["results"]["TS MSE"]
    TS_R2 = model_result["results"]["TS R2"]
    Nodes = model_result["results"]["Nodes"]
    
    timestamps = model_result["results"]["timestamp"]
    
    total_iterations = len(timestamps) - 1
    total_seconds = timestamps[-1] - timestamps[0]
    
    result = {}
    
    result["ground truth"] = ground_truth
    result["GSM"] = {"total iterations": total_iterations, "total seconds": total_seconds}
    
    add_information_performance(result["GSM"], last_model_index, "last", models, TS_MAE, TS_MSE, TS_R2, Nodes)
    add_information_performance(result["GSM"], best_model_index, "best", models, TS_MAE, TS_MSE, TS_R2, Nodes)
    
    result["competitors"] = {}
    
    competitors_results = pd.concat(
    [
        cavalab_bb_results[(cavalab_bb_results["dataset"] == dataset) & (cavalab_bb_results["random_state"] == shuffle_dataset_seed)],
        cavalab_gt_results[(cavalab_gt_results["dataset"] == dataset) & (cavalab_gt_results["random_state"] == shuffle_dataset_seed)]
    ],
    
    ignore_index=True)
    
    competitors = competitors_results["algorithm"].unique()
    
    for c in competitors:
        
        competitor_versions = competitors_results[competitors_results["algorithm"] == c]
        
        competitor_versions = competitor_versions.sort_values(by = "r2_test", ascending = False)
        
        result["competitors"][c] = {}
        
        symbolic_model = [str(x) for x in competitor_versions["symbolic_model"].to_numpy()]
        mae_test = competitor_versions["mae_test"].to_numpy()
        mse_test = competitor_versions["mse_test"].to_numpy()
        r2_test = competitor_versions["r2_test"].to_numpy()
        model_size = [str(x) for x in competitor_versions["model_size"].to_numpy()]
        training_time = competitor_versions["training time (s)"].to_numpy()
        
        add_information_performance(result["competitors"][c],  0, "worst", symbolic_model, mae_test, mse_test, r2_test, model_size)
        add_information_performance(result["competitors"][c], -1, "best",  symbolic_model, mae_test, mse_test, r2_test, model_size)
        
        result["competitors"][c] |= {"worst total seconds": training_time[0],
                                     "best total seconds": training_time[-1]}
    
    
    return result


result = {}

for dataset in os.listdir("BenchmarkCompleteResults"):
    
    dataset = dataset.split(".")[0]
    print(dataset)
    result[dataset] = informations(dataset)
    

json.dump(result, open(f"BenchmarkComparitions.json", "w", encoding="utf-8"), indent=4, ensure_ascii=False)

#ground truth (se c'è), sia per modello finale che per modello interno (con VL): MAE, MSE, R2 (solo test), complessità (numero di nodi)
#tempo di esecuzione di tutte le iterazioni + numero di iterazioni

1027_ESL
1028_SWD
1029_LEV
1030_ERA
1089_USCrime
1096_FacultySalaries
1193_BNG_lowbwt
1199_BNG_echoMonths
192_vineyard
197_cpu_act
201_pol
210_cloud
215_2dplanes
218_house_8L
225_puma8NH
227_cpu_small
228_elusage
229_pwLinear
230_machine_cpu
294_satellite_image
344_mv
4544_GeographicalOriginalofMusic
485_analcatdata_vehicle
503_wind
505_tecator
519_vinnie
522_pm10
523_analcatdata_neavote
527_analcatdata_election2000
529_pollen
537_houses
542_pollution
547_no2
556_analcatdata_apnea2
557_analcatdata_apnea1
560_bodyfat
561_cpu
562_cpu_small
564_fried
573_cpu_act
574_house_16H
579_fri_c0_250_5
581_fri_c3_500_25
582_fri_c1_500_25
583_fri_c1_1000_50
584_fri_c4_500_25
586_fri_c3_1000_25
588_fri_c4_1000_100
589_fri_c2_1000_25
590_fri_c0_1000_50
591_fri_c1_100_10
592_fri_c4_1000_25
593_fri_c1_1000_10
594_fri_c2_100_5
595_fri_c0_1000_10
596_fri_c2_250_5
597_fri_c2_500_5
598_fri_c0_1000_25
599_fri_c2_1000_5
601_fri_c1_250_5
602_fri_c3_250_10
603_fri_c0_250_50
604_fri_c4_500_10
605_fri_c2_250_25
6